# Tutorial 3: JitRL Full — Reward-Guided Logit Modulation with Hidden-State Embeddings

**Going beyond TF-IDF: encode documents as model hidden states for semantic retrieval.**

In the previous tutorial, JitRL MVP used TF-IDF (keyword matching) to retrieve relevant document chunks.
This works well when queries share exact vocabulary with the source documents, but fails on:

- **Paraphrased questions** — "Who founded the quantum division?" vs. "Who led the initiative?"
- **Semantic similarity** — questions about related concepts that use different words
- **Abstractive reasoning** — questions requiring synthesis across document sections

JitRL Full replaces TF-IDF with a **Knowledge Store** that encodes documents as the model's own
hidden-state embeddings, and uses a **Reward Computer** to modulate logits based on retrieval quality.

**In this tutorial you will learn:**
1. How the Knowledge Store encodes documents as hidden-state embeddings
2. How reward-guided modulation steers generation toward retrieved knowledge
3. When semantic matching outperforms keyword matching
4. How to compare MVP vs Full side-by-side

## Concept: The Knowledge Store

Instead of building a TF-IDF index over raw text, JitRL Full runs each document through the
frozen base model and stores the **last hidden-state embeddings**:

```
Document text  -->  Tokenize  -->  Forward pass (frozen model)  -->  Hidden states
                                                                        |
                                                         Mean-pool per document
                                                                        |
                                                              Knowledge Store
                                                         (list of embedding vectors)
```

At query time, the query is also encoded through the model, and **cosine similarity** finds
the closest stored documents. This captures semantic meaning rather than just shared words.

Key classes:
- `KnowledgeStore` — stores mean-pooled hidden states, supports `add()` and `query()`
- `RewardComputer` — computes element-wise reward from query-knowledge alignment
- `LogitModulator` — applies `logits + temperature * (reward @ projection)` during generation

## Concept: Reward-Guided Modulation

Once the Knowledge Store finds the most relevant documents, we need to steer generation.
JitRL Full does this via a **reward signal** that modulates logits at every decoding step:

1. **Compute reward**: `reward = normalize(query_mean) * normalize(knowledge_mean)` (element-wise)
2. **Project to vocabulary**: `reward_logits = reward @ lm_head.weight.T`
3. **Modulate**: `final_logits = original_logits + temperature * reward_logits`

The `modulation_temperature` controls how strongly the reward biases generation:
- **Low temperature (0.1)**: subtle nudge — model mostly follows its own priors
- **High temperature (2.0)**: strong bias — model heavily favors retrieved knowledge vocabulary
- **Default (0.5)**: balanced blend

This is implemented as a HuggingFace `LogitsProcessor` that plugs into `model.generate()`.

In [ ]:
# Cell 4: Load model and create JitRLFullEngine
import time
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32, trust_remote_code=True
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print(f"Model loaded on {device}")
print(f"Hidden size: {model.config.hidden_size}")

# Create the JitRL Full engine
from continual_learning.jitrl.full.engine import JitRLFullEngine

engine = JitRLFullEngine(
    model=model,
    tokenizer=tokenizer,
    modulation_temperature=0.5,  # How strongly to bias toward retrieved knowledge
    max_tokens=4096,
)
print(f"JitRLFullEngine created (temperature={engine.modulation_temperature})")

In [ ]:
# Cell 5: Learn from sample document and measure timing
doc_path = Path("data/sample_document.txt")
document = doc_path.read_text()
print(f"Document: {len(document.split())} words")
print(f"Preview: {document[:200]}...\n")

# Learn with JitRL Full (requires a forward pass to compute hidden states)
t0 = time.time()
result = engine.learn(document)
full_time = time.time() - t0

print(f"JitRL Full learn() completed in {full_time:.3f}s")
print(f"Tokens processed: {result['tokens_processed']}")
print(f"Method: {result['method']}")
print(f"Knowledge store entries: {engine._knowledge_store.num_entries}")

# Compare: JitRL MVP learn is near-instant (TF-IDF indexing only)
from continual_learning.jitrl.mvp.engine import JitRLMVPEngine

mvp_engine = JitRLMVPEngine(model=model, tokenizer=tokenizer)
t0 = time.time()
mvp_result = mvp_engine.learn(document)
mvp_time = time.time() - t0

print(f"\nJitRL MVP learn() completed in {mvp_time:.3f}s")
print(f"Full is ~{full_time / max(mvp_time, 0.001):.1f}x slower (due to model forward pass)")

In [ ]:
# Cell 6: Query with questions and inspect the knowledge store
questions = [
    "Who led the Quantum Computing Research Division at Oracle Labs?",
    "What was the RedShift processor's quantum volume?",
    "When was Lattice Shield integrated into OCI?",
]

print("=" * 70)
print("QUERYING JitRL Full Engine")
print("=" * 70)
for q in questions:
    response = engine.generate(q, max_new_tokens=100)
    print(f"\nQ: {q}")
    print(f"A: {response[:300]}")

# Inspect the knowledge store internals
print("\n" + "=" * 70)
print("KNOWLEDGE STORE INTERNALS")
print("=" * 70)
ks = engine._knowledge_store
print(f"Number of entries: {ks.num_entries}")
print(f"Embedding dimension: {ks.hidden_size}")
for i, (name, emb) in enumerate(zip(ks._doc_names, ks._doc_embeddings)):
    print(f"  [{i}] {name}: shape={emb.shape}, norm={emb.norm():.4f}, mean={emb.mean():.6f}")

## MVP vs Full: When Does Semantic Matching Win?

TF-IDF (MVP) works by matching **exact words** between the query and stored documents.
Hidden-state embeddings (Full) capture **semantic meaning** through the model's learned representations.

| Scenario | MVP (TF-IDF) | Full (Hidden States) |
|----------|-------------|---------------------|
| Query uses same words as document | Excellent | Good |
| Paraphrased query | Poor | Good |
| Cross-document reasoning | Poor | Better |
| Speed (learn) | ~0.002s | ~0.5-2s |
| Speed (generate) | Fast | Slightly slower |
| Memory | TF-IDF index | Hidden-state tensors |

The key insight: **if your users will ask questions using different vocabulary than the source
documents, Full's semantic matching is worth the extra cost.**

In [ ]:
# Cell 8: Test with paraphrased questions — does Full outperform MVP?
# These questions deliberately use different vocabulary from the source document
paraphrased_questions = [
    {
        "question": "Who was the founding leader of Oracle's quantum research effort?",
        "answer": "Dr. Sarah Chen",
        "note": "Document says 'led by Dr. Sarah Chen' — this uses 'founding leader'",
    },
    {
        "question": "What encryption method was developed with NIST to protect against quantum threats?",
        "answer": "Lattice Shield",
        "note": "Document says 'post-quantum cryptography' — this says 'quantum threats'",
    },
    {
        "question": "How many researchers were in the quantum team by end of 2024?",
        "answer": "ninety-five",
        "note": "Document says 'headcount grew to ninety-five'",
    },
]

print("Paraphrased Question Test: MVP vs Full")
print("=" * 70)

mvp_correct = 0
full_correct = 0

for item in paraphrased_questions:
    q = item["question"]
    expected = item["answer"].lower()

    mvp_answer = mvp_engine.generate(f"Answer concisely: {q}", max_new_tokens=80)
    full_answer = engine.generate(f"Answer concisely: {q}", max_new_tokens=80)

    mvp_hit = expected in mvp_answer.lower()
    full_hit = expected in full_answer.lower()
    mvp_correct += int(mvp_hit)
    full_correct += int(full_hit)

    print(f"\nQ: {q}")
    print(f"  Expected: {item['answer']}")
    print(f"  Note: {item['note']}")
    print(f"  MVP:  {'CORRECT' if mvp_hit else 'WRONG'} — {mvp_answer[:150]}")
    print(f"  Full: {'CORRECT' if full_hit else 'WRONG'} — {full_answer[:150]}")

print(f"\nScore: MVP {mvp_correct}/{len(paraphrased_questions)}, "
      f"Full {full_correct}/{len(paraphrased_questions)}")

In [ ]:
# Cell 9: Side-by-side comparison using ComparisonHarness
from continual_learning.jitrl.comparison import ComparisonHarness

# Fresh engines for a clean comparison
mvp_fresh = JitRLMVPEngine(model=model, tokenizer=tokenizer)
full_fresh = JitRLFullEngine(
    model=model, tokenizer=tokenizer, modulation_temperature=0.5
)

harness = ComparisonHarness(engines={"MVP": mvp_fresh, "Full": full_fresh})

qa_items = [
    {"question": "Who led the quantum division?", "answer": "Dr. Sarah Chen"},
    {"question": "What was the code name of the 72-qubit processor?", "answer": "RedShift"},
    {"question": "What framework simulated quantum circuits on classical hardware?", "answer": "QubitFlow"},
    {"question": "What was the division's annual budget?", "answer": "$120 million"},
    {"question": "What year did the quantum division start?", "answer": "2019"},
]

results = harness.run_comparison(
    learn_texts=[document],
    qa_items=qa_items,
    max_new_tokens=80,
)

print(f"{'Metric':<25} {'MVP':>12} {'Full':>12}")
print("-" * 50)
for metric in ["accuracy", "correct", "num_evaluated", "learn_time_s", "eval_time_s"]:
    mvp_val = results["MVP"][metric]
    full_val = results["Full"][metric]
    if isinstance(mvp_val, float):
        print(f"{metric:<25} {mvp_val:>12.3f} {full_val:>12.3f}")
    else:
        print(f"{metric:<25} {mvp_val:>12} {full_val:>12}")

## When to Use Full vs MVP

**Choose JitRL MVP when:**
- Speed is critical (near-instant learning)
- Queries use similar vocabulary to source documents
- You have many documents and need lightweight indexing
- Running on CPU-only hardware

**Choose JitRL Full when:**
- Users ask paraphrased or semantically complex questions
- You need better cross-document reasoning
- You have GPU available (forward pass is faster on GPU)
- Quality matters more than latency

**Key trade-off**: Full pays a one-time cost at `learn()` (model forward pass to compute embeddings)
but gains semantic understanding. MVP pays nothing at learn time but is limited to keyword matching.

## Exercises

1. **Temperature sweep**: Try `modulation_temperature` values of 0.1, 0.5, 1.0, and 2.0.
   How does the strength of the reward modulation affect answer quality? At what point does
   high temperature cause the model to generate incoherent text?

2. **Multi-document retrieval**: Add 3-4 different documents to the Knowledge Store, then
   query about each. Inspect `_knowledge_store.query()` to verify the correct document
   gets the highest cosine similarity score.

3. **Embedding visualization**: Extract the document embeddings from `_knowledge_store._doc_embeddings`,
   reduce to 2D with PCA or t-SNE, and plot them. Do semantically similar documents cluster together?

4. **Token-level analysis**: Use `_knowledge_store.get_token_embeddings(index)` to examine
   per-token hidden states. Which tokens have the highest norm? Do they correspond to
   important content words?

5. **Scaling test**: Measure learn time and query time as you add 1, 5, 10, 20 documents.
   How does the Knowledge Store's cosine similarity search scale?

In [ ]:
# Cell 12: Clean up
engine.clear()
mvp_engine.clear()
print("Engines cleared.")
print(f"Knowledge store entries after clear: {engine._knowledge_store.num_entries}")